# AST Audio Spectrogram Transformer — DIMER Acoustic Ecology E2E Tutorial (Standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/ast-audio-classification-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/ast-audio-classification-pipeline/blob/main/tutorials/ast_audio_classification_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-MIT%2Fast--finetuned--audioset-ffcc4d?style=flat)](https://huggingface.co/MIT/ast-finetuned-audioset-10-10-0.4593) [![Upstream](https://img.shields.io/badge/Upstream-YuanGongND%2Fast-181717?style=flat&logo=github&logoColor=white)](https://github.com/YuanGongND/ast) [![arXiv](https://img.shields.io/badge/arXiv-2104.01778-b31b1b.svg)](https://arxiv.org/abs/2104.01778)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** pretrained AudioSet classification & supervised in-kernel acoustic ecology adaptation with `MIT/ast-finetuned-audioset-10-10-0.4593`

**This notebook is standalone.** It carries the repository's package (3 modules under `src/ast_audio_classification_pipeline/`, at revision `c54ddcf12996`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `f826b80d28226b62986cc218e5cec390b1096902` (~346 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned AST snapshot, demonstrates the unchanged 527-label AudioSet head, generates and validates the deterministic 24-clip acoustic-ecology dataset, creates a seeded disjoint split, re-heads the classifier onto three classes, freezes the transformer backbone, measures the pre-adaptation baseline, runs the bounded classifier-head fine-tune, evaluates the held-out clips, predicts an unseen generated clip, exports the classifier-head adapter, reloads it over a fresh base-model instance, verifies numeric parity, and writes machine-readable outputs. The default path needs no repository clone, DIMER worker or service, credential, upload dialog, or configuration edit (NOTEBOOK_SPEC 2.0 §5, RUN7, FT2).

**Bring Your Own Data:** Two optional branches are disabled by default. `USE_BYOD = True` accepts one 16 kHz PCM WAV for the unchanged AudioSet inference demonstration. `USE_BYOD_DATASET = True` accepts one ZIP whose top-level directories are exactly `geophony/`, `biophony/`, and `anthrophony/`, with at least two 16 kHz mono or multichannel PCM WAV files per class; each clip must be 0.025–10.24 s. The ZIP branch applies explicit archive limits, decodes to mono float32, then enters the same validation, seeded split, local adaptation, evaluation, export, and reload path as the generated dataset. Uploads stay in this runtime.

The Audio Spectrogram Transformer (AST) applies a Vision Transformer (ViT) architecture directly to audio spectrograms. At inference, input audio is resampled to 16 kHz, converted into a 128-bin Kaldi filterbank with a 10 ms hop, padded or cropped to 1024 frames (10.24 s), and processed by the transformer backbone. **In-kernel fine-tuning:** this tutorial demonstrates both pretrained 527-class AudioSet event inference and end-to-end supervised adaptation to a custom acoustic ecology classification task (`ADAPT_CLASSES = ('geophony', 'biophony', 'anthrophony')`). The backbone is frozen and the classifier head is dynamically re-headed (`Linear(768, 3)`), training only ~3.8k parameters with bounded AdamW in ~10–15 s on CPU with zero external worker repositories. Quantitative evaluation against a majority-class baseline and portable adapter export (`outputs/ast-audio-adapter-v1.pt`) complete the end-to-end adaptation workflow.

**Learning objectives:** install the pinned runtime, verify the immutable upstream AST checkpoint against its SHA-256 manifest, demonstrate pretrained AudioSet inference on a synthetic tone, generate and validate an in-code acoustic ecology dataset conforming to the owner-namespaced `io.github.kurtvalcorza.dataset.audio.waveform-classification.v1` representation, perform a stratified 75/25 split, dynamically re-head the classifier and freeze the backbone, execute an in-process AdamW fine-tuning loop, quantitatively evaluate accuracy and macro-F1 against a majority baseline, export a portable adapter artifact, and verify safe reloading across an isolated boundary.

**This notebook does not demonstrate:** speech transcription, speaker identification, temporal localisation of events inside the window, source separation, or unconstrained multi-hour training. Adaptation is bounded to classifier head adaptation.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). Runs on CPU or CUDA automatically; float32 on both.
- **Knowledge:** basic Python and PyTorch; familiarity with acoustic spectrograms and classification baselines.
- **Data:** the default path is 100% self-contained. Optional single-clip BYOD accepts one PCM WAV. Optional adaptation BYOD accepts one ZIP with exactly `geophony/`, `biophony/`, and `anthrophony/` top-level directories and at least two 16 kHz PCM WAV files per class; clips must be 0.025–10.24 s. The ZIP is capped at 100 files, 64 MiB compressed, and 256 MiB expanded. Do not upload confidential or restricted audio to a hosted runtime unless authorized.
- **External access:** the Hugging Face Hub only, to fetch the pinned `MIT/ast-finetuned-audioset-10-10-0.4593` snapshot (~346 MB in total) at revision `f826b80d2822…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `torchaudio`, `transformers` versions, and whether CUDA is available.

In [1]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'ast-audio-classification-pipeline',
    'repository_revision': 'c54ddcf129968b835ef2ffda2bf5da8478a1a435',
    'embedded_module': 'src/ast_audio_classification_pipeline/pipeline.py',
    'embedded_modules': ['src/ast_audio_classification_pipeline/metrics.py', 'src/ast_audio_classification_pipeline/pipeline.py', 'src/ast_audio_classification_pipeline/samples.py'],
    'module_sha256': 'e6c6c76092853868e8aa69880b874e124aca59ac5970cbca8e369757c9df15c6',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, torchaudio, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'torchaudio': torchaudio.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 67.8 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 29.3 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.0 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 3.4 MB/s eta 0:00:00


...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/145.9 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.0/148.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 87.2 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 4.9 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 MB 42.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
libcuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have 

RuntimeError: Core dependencies changed while older modules were loaded: cuda-bindings: loaded=12.9.4, installed=13.4.1; numpy: loaded=2.0.2, installed=2.5.3. Restart the runtime, then rerun from the top.

## 2. Pipeline code (carried verbatim from `src/ast_audio_classification_pipeline/` @ `c54ddcf12996`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/ast_audio_classification_pipeline/metrics.py`

In [ ]:
"""Evaluation metrics for multiclass audio classification."""

from __future__ import annotations

from collections import Counter
from collections.abc import Sequence
from typing import Any


def _validate_num_classes(num_classes: int) -> None:
    if isinstance(num_classes, bool) or not isinstance(num_classes, int) or num_classes < 1:
        raise ValueError("num_classes must be a positive int")


def _validate_class_indices(values: Sequence[int], num_classes: int, *, name: str) -> None:
    for value in values:
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{name} must contain ints, got {type(value).__name__}")
        if not 0 <= value < num_classes:
            raise ValueError(f"{name} class index {value} outside [0, {num_classes - 1}]")


def multiclass_accuracy(predictions: Sequence[int], targets: Sequence[int]) -> float:
    """Compute overall accuracy as the fraction of correctly classified clips."""
    if len(predictions) != len(targets):
        raise ValueError(f"length mismatch: predictions={len(predictions)}, targets={len(targets)}")
    if len(targets) == 0:
        raise ValueError("cannot compute accuracy over empty targets")
    correct = sum(1 for p, t in zip(predictions, targets, strict=True) if p == t)
    return correct / len(targets)


def confusion_matrix(
    predictions: Sequence[int], targets: Sequence[int], num_classes: int
) -> list[list[int]]:
    """Compute confusion matrix where rows are ground truth classes and columns are predicted classes."""
    _validate_num_classes(num_classes)
    if len(predictions) != len(targets):
        raise ValueError(f"length mismatch: predictions={len(predictions)}, targets={len(targets)}")
    _validate_class_indices(predictions, num_classes, name="predictions")
    _validate_class_indices(targets, num_classes, name="targets")
    matrix = [[0] * num_classes for _ in range(num_classes)]
    for p, t in zip(predictions, targets, strict=True):
        matrix[t][p] += 1
    return matrix


def per_class_metrics(
    predictions: Sequence[int], targets: Sequence[int], num_classes: int
) -> dict[int, dict[str, float]]:
    """Compute per-class precision, recall, and F1-score."""
    if len(predictions) != len(targets):
        raise ValueError(f"length mismatch: predictions={len(predictions)}, targets={len(targets)}")
    matrix = confusion_matrix(predictions, targets, num_classes)
    results: dict[int, dict[str, float]] = {}
    for c in range(num_classes):
        tp = matrix[c][c]
        fp = sum(matrix[other][c] for other in range(num_classes) if other != c)
        fn = sum(matrix[c][other] for other in range(num_classes) if other != c)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2.0 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        results[c] = {
            "precision": round(float(precision), 4),
            "recall": round(float(recall), 4),
            "f1": round(float(f1), 4),
            "support": sum(matrix[c]),
        }
    return results


def macro_f1_score(
    predictions: Sequence[int], targets: Sequence[int], num_classes: int
) -> float:
    """Compute unweighted macro-averaged F1 score across all classes."""
    pcm = per_class_metrics(predictions, targets, num_classes)
    f1_sum = sum(metrics["f1"] for metrics in pcm.values())
    return round(float(f1_sum / num_classes), 4) if num_classes > 0 else 0.0


def majority_class_baseline(targets: Sequence[int], num_classes: int) -> dict[str, Any]:
    """Compute the accuracy of a constant trivial classifier predicting the majority class."""
    _validate_num_classes(num_classes)
    if len(targets) == 0:
        raise ValueError("cannot compute baseline over empty targets")
    _validate_class_indices(targets, num_classes, name="targets")
    counts = Counter(targets)
    majority_class, count = counts.most_common(1)[0]
    majority_acc = count / len(targets)
    return {
        "majority_class_index": int(majority_class),
        "majority_class_count": int(count),
        "majority_class_accuracy": round(float(majority_acc), 4),
        "num_classes": num_classes,
        "n_samples": len(targets),
    }


def evaluate_classification(
    predictions: Sequence[int],
    targets: Sequence[int],
    class_names: Sequence[str],
) -> dict[str, Any]:
    """Comprehensive multi-class classification evaluation against baselines."""
    num_classes = len(class_names)
    if num_classes < 1:
        raise ValueError("class_names must not be empty")
    if any(not isinstance(name, str) or not name.strip() for name in class_names):
        raise ValueError("class_names must contain non-empty strings")
    if len(set(class_names)) != num_classes:
        raise ValueError("class_names must be unique")
    acc = multiclass_accuracy(predictions, targets)
    pcm = per_class_metrics(predictions, targets, num_classes)
    macro_f1 = macro_f1_score(predictions, targets, num_classes)
    baseline = majority_class_baseline(targets, num_classes)
    cm = confusion_matrix(predictions, targets, num_classes)

    named_per_class = {class_names[c]: pcm[c] for c in range(num_classes)}

    return {
        "accuracy": round(float(acc), 4),
        "macro_f1": macro_f1,
        "per_class": named_per_class,
        "confusion_matrix": cm,
        "baseline": baseline,
        "accuracy_delta_vs_baseline": round(float(acc - baseline["majority_class_accuracy"]), 4),
        "num_samples": len(targets),
        "num_classes": num_classes,
    }

**Module 2/3:** `src/ast_audio_classification_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np

MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"
MODEL_REVISION = "f826b80d28226b62986cc218e5cec390b1096902"
MODEL_LICENSE = "bsd-3-clause"
MODEL_KEY = "ast-audioset"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# The pinned checkpoint expects 16 kHz mono float32. Its feature extractor computes a 128-bin Kaldi
# fbank with a 10 ms hop and pads or crops to 1024 frames, so only the first 10.24 s of audio reach
# the model; anything longer is cropped and reported as `truncated`.
SAMPLE_RATE = 16_000
WINDOW_FRAMES = 1024
MAX_AUDIO_SECONDS = WINDOW_FRAMES * 0.010  # 10.24 s model window
MAX_INPUT_SECONDS = 120.0  # hard ceiling: longer input is rejected, the caller must chunk
MIN_AUDIO_SECONDS = 0.025  # one 25 ms fbank frame
NUM_LABELS = 527  # AudioSet ontology classes in the pinned config.json
DEFAULT_TOP_K = 5
ACTIVATION = "sigmoid"
ARTIFACT_FORMAT = "org.valcorza.ast-audio.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = hashlib.sha256()
        with open(file_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                digest.update(chunk)
        if digest.hexdigest() != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest.hexdigest()} != manifest {entry['sha256']}")
    return manifest


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _resample(waveform: np.ndarray, sample_rate: int) -> np.ndarray:
    import torch
    import torchaudio.functional as af

    resampled = af.resample(torch.from_numpy(waveform), orig_freq=sample_rate, new_freq=SAMPLE_RATE)
    return resampled.numpy().astype(np.float32)


INPUT_SCHEMA: dict[str, Any] = {
    "input": "1-D float numpy array of mono samples in [-1, 1], plus the sample_rate it was captured at",
    "sample_rate_hz": f"any positive int; resampled to SAMPLE_RATE={SAMPLE_RATE} when it differs",
    "duration_seconds": [MIN_AUDIO_SECONDS, MAX_INPUT_SECONDS],
    "model_window_seconds": MAX_AUDIO_SECONDS,
    "top_k": [1, NUM_LABELS],
    "preprocessing": (
        f"resample to {SAMPLE_RATE} Hz when needed, 128-bin Kaldi fbank with a 10 ms hop, "
        f"pad or crop to {WINDOW_FRAMES} frames ({MAX_AUDIO_SECONDS} s)"
    ),
}


def _check_inputs(audio: Any, sample_rate: Any, top_k: Any) -> float:
    """Raise TypeError/ValueError naming the first violated ceiling; return the clip duration in seconds."""
    if not isinstance(audio, np.ndarray):
        raise TypeError(f"audio must be a numpy.ndarray, got {type(audio).__name__}")
    if audio.ndim != 1:
        raise ValueError(f"audio must be 1-D mono, got shape {audio.shape}")
    if not np.issubdtype(audio.dtype, np.floating):
        raise TypeError(f"audio must be a float array in [-1, 1], got dtype {audio.dtype}")
    if not isinstance(sample_rate, int) or isinstance(sample_rate, bool) or sample_rate <= 0:
        raise TypeError("sample_rate must be a positive int (the rate the audio was captured at)")
    if not isinstance(top_k, int) or isinstance(top_k, bool) or not 1 <= top_k <= NUM_LABELS:
        raise ValueError(f"top_k must be an int in [1, {NUM_LABELS}]")
    duration = audio.shape[0] / sample_rate
    if duration < MIN_AUDIO_SECONDS:
        raise ValueError(f"audio is {duration:.4f} s; minimum is {MIN_AUDIO_SECONDS} s")
    if duration > MAX_INPUT_SECONDS:
        raise ValueError(f"audio is {duration:.2f} s; ceiling is {MAX_INPUT_SECONDS} s (chunk it first)")
    if not np.all(np.isfinite(audio)):
        raise ValueError("audio contains NaN or inf samples")
    if np.max(np.abs(audio)) > 1.0:
        raise ValueError("audio samples must be in [-1, 1]")
    return duration


def validate_inputs(
    waveforms: Any,
    sample_rate: Any,
    *,
    top_k: int = DEFAULT_TOP_K,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-clip observations, verdict).

    The checks are the ones ``predict`` applies, through the same private ``_check_inputs``, so a
    rejection here raises exactly what ``predict`` would; a caller that wants the finding recorded
    catches the exception and stores ``str(exc)`` under ``findings``.
    """
    batch = [waveforms] if isinstance(waveforms, np.ndarray) else waveforms
    if not isinstance(batch, Sequence) or isinstance(batch, str | bytes):
        raise TypeError("waveforms must be a 1-D numpy.ndarray or a sequence of them")
    if len(batch) < 1:
        raise ValueError("at least one waveform is required")
    if names is not None and len(names) != len(batch):
        raise ValueError("names must have one entry per waveform")
    inputs = []
    for index, waveform in enumerate(batch):
        duration = _check_inputs(waveform, sample_rate, top_k)
        inputs.append(
            {
                "id": names[index] if names else f"clip-{index}",
                "samples": int(waveform.shape[0]),
                "dtype": str(waveform.dtype),
                "duration_seconds": round(duration, 4),
                "peak_amplitude": round(float(np.max(np.abs(waveform))), 6),
                "will_resample": sample_rate != SAMPLE_RATE,
                "will_truncate": duration > MAX_AUDIO_SECONDS,
            }
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": inputs,
        "sample_rate": sample_rate,
        "top_k": top_k,
        "n_clips": len(inputs),
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    targets: Sequence[Any] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Describe the unadapted AudioSet inference result without inventing ground truth."""
    predictions = result["predictions"]
    reason = "no AudioSet-labelled ground truth exists for the evaluated clip"
    if targets is not None:
        reason = (
            "targets were supplied, but they are not established as labels from the 527-class "
            "AudioSet ontology; "
            "score only ontology-aligned labels with an AudioSet-convention mAP implementation"
        )
    return {
        "task": "multi-label audio event classification over the 527 AudioSet labels",
        "score_semantics": (
            f"independent {result.get('activation', ACTIVATION)} score per label: the scores do not sum "
            "to one, several labels can be high at once, none is a calibrated probability, and no "
            "threshold is shipped"
        ),
        "sample_kind": sample_kind,
        "n_clips": 1,
        "n_scored_labels": len(predictions),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": reason,
        "needs": (
            f"clips labelled against the same {NUM_LABELS}-label AudioSet ontology, scored over the full "
            "score vector (predict(..., top_k=527)) with mean average precision plus per-label precision "
            "and recall at a threshold chosen on your own labelled clips; the '0.4593' in the checkpoint "
            "name is the upstream-reported AudioSet mAP and is not measured here"
        ),
        "truncated": bool(result.get("truncated", False)),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def rehead_model(
    model: Any,
    class_names: Sequence[str],
    seed: int = 42,
) -> None:
    """Dynamically replace ASTMLPHead dense layer with a new linear classifier sized to class_names."""
    import torch
    import torch.nn as nn

    names = [str(name).strip() for name in class_names]
    if len(names) < 2 or any(not name for name in names):
        raise ValueError("class_names must contain at least 2 non-empty names")
    if len(set(names)) != len(names):
        raise ValueError("class_names must be unique")
    previous_dense = model.classifier.dense
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed)
        new_dense = nn.Linear(previous_dense.in_features, len(names), bias=True)
        nn.init.kaiming_normal_(new_dense.weight, nonlinearity="linear")
        nn.init.zeros_(new_dense.bias)
    new_dense = new_dense.to(device=previous_dense.weight.device, dtype=previous_dense.weight.dtype)
    model.classifier.dense = new_dense
    model.num_labels = len(names)
    model.config.num_labels = len(names)
    model.config.id2label = dict(enumerate(names))
    model.config.label2id = {name: i for i, name in enumerate(names)}
    model.config.problem_type = "single_label_classification"


def freeze_backbone(model: Any) -> int:
    """Freeze all parameters in the audio spectrogram transformer backbone, returning frozen count."""
    frozen = 0
    for param in model.audio_spectrogram_transformer.parameters():
        param.requires_grad = False
        frozen += param.numel()
    return frozen


@dataclass
class ASTAudioClassificationPipeline:
    """Audio Spectrogram Transformer classifier for AudioSet and adapted audio tasks."""

    _runner: Callable[[np.ndarray], np.ndarray]
    labels: list[str]
    device: str
    model: Any | None = None
    extractor: Any | None = None
    activation: str = ACTIVATION
    adaptation_config: dict[str, Any] = field(default_factory=dict)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> ASTAudioClassificationPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), dict(local_files_only=True)
        elif allow_download:
            source, kwargs = MODEL_ID, dict(revision=MODEL_REVISION)
        else:
            raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import ASTFeatureExtractor, ASTForAudioClassification

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        extractor = ASTFeatureExtractor.from_pretrained(source, trust_remote_code=False, **kwargs)
        model = ASTForAudioClassification.from_pretrained(
            source, trust_remote_code=False, dtype=torch.float32, **kwargs
        )
        model = model.to(resolved_device).eval()
        labels = [model.config.id2label[i] for i in range(model.config.num_labels)]

        def runner(waveform: np.ndarray) -> np.ndarray:
            inputs = extractor(waveform, sampling_rate=SAMPLE_RATE, return_tensors="pt")
            with torch.inference_mode():
                logits = model(inputs["input_values"].to(resolved_device)).logits
            return logits[0].float().cpu().numpy()

        return cls(runner, labels, resolved_device, model=model, extractor=extractor)

    def rehead(self, class_names: Sequence[str], seed: int = 42) -> None:
        """Dynamically rehead the underlying model and update labels and runner."""
        if self.model is None or self.extractor is None:
            raise RuntimeError("cannot rehead a pipeline instance without an underlying torch model")
        rehead_model(self.model, class_names, seed=seed)
        self.labels = [self.model.config.id2label[i] for i in range(self.model.config.num_labels)]
        self.activation = "softmax"
        self.adaptation_config = {
            "mode": "classifier-head-gradient-adaptation",
            "class_names": list(self.labels),
            "rehead_seed": seed,
        }
        self.model.to(self.device)
        self._refresh_runner()

    def _refresh_runner(self) -> None:
        """Bind inference to the current model, extractor, and device."""
        if self.model is None or self.extractor is None:
            raise RuntimeError("cannot bind a runner without an underlying model and extractor")
        resolved_device = self.device
        model = self.model
        extractor = self.extractor

        def runner(waveform: np.ndarray) -> np.ndarray:
            import torch

            inputs = extractor(waveform, sampling_rate=SAMPLE_RATE, return_tensors="pt")
            with torch.inference_mode():
                logits = model(inputs["input_values"].to(resolved_device)).logits
            return logits[0].float().cpu().numpy()

        self._runner = runner

    def freeze_backbone(self) -> int:
        """Freeze the underlying backbone parameters."""
        if self.model is None:
            raise RuntimeError(
                "cannot freeze backbone on a pipeline instance without an underlying torch model"
            )
        frozen = freeze_backbone(self.model)
        self.adaptation_config["frozen_backbone_parameters"] = frozen
        return frozen

    def finetune(
        self,
        train_records: Sequence[dict[str, Any]],
        val_records: Sequence[dict[str, Any]] | None = None,
        *,
        epochs: int = 3,
        batch_size: int = 4,
        learning_rate: float = 1e-4,
        seed: int = 42,
    ) -> list[dict[str, Any]]:
        """In-process supervised fine-tuning using AdamW optimizer over trainable classifier parameters."""
        import random

        import torch
        import torch.nn.functional as F
        from torch.optim import AdamW

        pass  # standalone rewrite (build_notebook.py): `from .metrics import evaluate_classification` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if self.model is None or self.extractor is None:
            raise RuntimeError("cannot fine-tune a pipeline without underlying model and extractor")
        if isinstance(epochs, bool) or not isinstance(epochs, int) or epochs < 1:
            raise ValueError("epochs must be a positive int")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or batch_size < 1:
            raise ValueError("batch_size must be a positive int")
        valid_learning_rate = isinstance(learning_rate, int | float) and not isinstance(learning_rate, bool)
        if not valid_learning_rate or learning_rate <= 0:
            raise ValueError("learning_rate must be positive")
        validate_dataset(train_records, self.labels)
        if val_records is not None:
            validate_dataset(val_records, self.labels)
        if any(param.requires_grad for param in self.model.audio_spectrogram_transformer.parameters()):
            raise RuntimeError("backbone is trainable; call freeze_backbone() before finetune()")

        torch.manual_seed(seed)
        device = torch.device(self.device)
        self.model.to(device)

        trainable_params = [p for p in self.model.parameters() if p.requires_grad]
        if not trainable_params:
            raise RuntimeError("model has no trainable parameters")
        optimizer = AdamW(trainable_params, lr=learning_rate)

        def cached_features(
            records: Sequence[dict[str, Any]],
        ) -> tuple[torch.Tensor, torch.Tensor]:
            feature_batches: list[torch.Tensor] = []
            self.model.audio_spectrogram_transformer.eval()
            with torch.inference_mode():
                for start in range(0, len(records), batch_size):
                    batch = records[start : start + batch_size]
                    inputs = self.extractor(
                        [record["waveform"] for record in batch],
                        sampling_rate=SAMPLE_RATE,
                        return_tensors="pt",
                    )
                    backbone_output = self.model.audio_spectrogram_transformer(
                        inputs["input_values"].to(device)
                    )
                    feature_batches.append(backbone_output.pooler_output.detach())
            targets = torch.tensor(
                [record["label"] for record in records],
                dtype=torch.long,
                device=device,
            )
            return torch.cat(feature_batches), targets

        train_features, train_targets = cached_features(train_records)
        val_features: torch.Tensor | None = None
        val_targets: torch.Tensor | None = None
        if val_records:
            val_features, val_targets = cached_features(val_records)

        history: list[dict[str, Any]] = []
        n_samples = len(train_records)

        for epoch in range(1, epochs + 1):
            self.model.classifier.train()
            indices = list(range(n_samples))
            rng = random.Random(seed + epoch * 13)
            rng.shuffle(indices)

            running_loss = 0.0
            steps = 0

            for i in range(0, n_samples, batch_size):
                batch_indices = indices[i : i + batch_size]
                index_tensor = torch.tensor(batch_indices, dtype=torch.long, device=device)

                optimizer.zero_grad(set_to_none=True)
                logits = self.model.classifier(train_features.index_select(0, index_tensor))
                loss = F.cross_entropy(logits, train_targets.index_select(0, index_tensor))
                loss.backward()
                optimizer.step()

                running_loss += float(loss.item())
                steps += 1

            epoch_train_loss = running_loss / steps if steps > 0 else 0.0
            epoch_data: dict[str, Any] = {
                "epoch": epoch,
                "train_loss": round(float(epoch_train_loss), 4),
                "steps": steps,
            }

            if val_features is not None and val_targets is not None:
                self.model.classifier.eval()
                with torch.inference_mode():
                    predictions = self.model.classifier(val_features).argmax(dim=-1)
                eval_metrics = evaluate_classification(
                    predictions.tolist(), val_targets.tolist(), self.labels
                )
                epoch_data["val_accuracy"] = eval_metrics["accuracy"]
                epoch_data["val_macro_f1"] = eval_metrics["macro_f1"]

            history.append(epoch_data)

        self.model.eval()
        self.adaptation_config.update(
            {
                "epochs": epochs,
                "batch_size": batch_size,
                "learning_rate": float(learning_rate),
                "training_seed": seed,
                "train_records": len(train_records),
                "validation_records": len(val_records or []),
                "feature_cache": "frozen-backbone-pooler-output",
                "history": history,
            }
        )
        return history

    def evaluate(self, records: Sequence[dict[str, Any]]) -> dict[str, Any]:
        """Evaluate the pipeline on a sequence of labelled audio records."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import evaluate_classification` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        validate_dataset(records, self.labels)
        if self.model is not None:
            self.model.eval()
        predictions: list[int] = []
        targets: list[int] = []

        for record in records:
            waveform = record["waveform"]
            sample_rate = record.get("sample_rate", SAMPLE_RATE)
            result = self.predict(waveform, sample_rate=sample_rate, top_k=1)
            pred_idx = result["predictions"][0]["index"]
            predictions.append(pred_idx)
            targets.append(record["label"])

        return evaluate_classification(predictions, targets, self.labels)

    def save_artifact(self, output_path: str | Path) -> Path:
        """Export adapted weights, vocabulary, base model lineage, and artifact metadata."""
        import torch

        if self.model is None:
            raise RuntimeError("cannot save artifact without an underlying torch model")

        out = Path(output_path)
        out.parent.mkdir(parents=True, exist_ok=True)

        if self.activation != "softmax" or len(self.labels) == NUM_LABELS:
            raise RuntimeError("artifact export requires an adapted single-label classifier")

        payload = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "artifact_kind": "classifier-head-adapter",
            "base_model_id": MODEL_ID,
            "base_model_revision": MODEL_REVISION,
            "class_names": list(self.labels),
            "num_classes": len(self.labels),
            "activation": self.activation,
            "adaptation": dict(self.adaptation_config),
            "classifier_state_dict": self.model.classifier.state_dict(),
        }
        torch.save(payload, out)
        return out

    def load_artifact(self, artifact_path: str | Path) -> None:
        """Load an adapted artifact into the existing pipeline safely using weights_only=True."""
        import torch

        if self.model is None or self.extractor is None:
            raise RuntimeError("cannot load artifact into a pipeline without an underlying torch model")

        payload = torch.load(artifact_path, map_location=self.device, weights_only=True)
        if not isinstance(payload, dict):
            raise ValueError(f"expected artifact dict, got {type(payload).__name__}")
        if payload.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"unrecognized artifact format: {payload.get('format')}")
        if payload.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(f"unsupported artifact format_version: {payload.get('format_version')}")
        if payload.get("artifact_kind") != "classifier-head-adapter":
            raise ValueError(f"unsupported artifact_kind: {payload.get('artifact_kind')}")
        if payload.get("base_model_id") != MODEL_ID:
            raise ValueError(f"base_model_id mismatch: {payload.get('base_model_id')} != {MODEL_ID}")
        if payload.get("base_model_revision") != MODEL_REVISION:
            raise ValueError(
                f"base_model_revision mismatch: {payload.get('base_model_revision')} != {MODEL_REVISION}"
            )
        if payload.get("activation") != "softmax":
            raise ValueError(f"unsupported artifact activation: {payload.get('activation')}")

        class_names = payload.get("class_names")
        if not isinstance(class_names, list) or payload.get("num_classes") != len(class_names):
            raise ValueError("artifact class_names and num_classes are inconsistent")
        rehead_model(self.model, class_names)
        classifier_state = payload.get("classifier_state_dict")
        if not isinstance(classifier_state, dict):
            raise ValueError("artifact classifier_state_dict must be a dict")
        self.model.classifier.load_state_dict(classifier_state, strict=True)
        self.labels = list(class_names)
        self.activation = "softmax"
        adaptation = payload.get("adaptation", {})
        if not isinstance(adaptation, dict):
            raise ValueError("artifact adaptation metadata must be a dict")
        self.adaptation_config = dict(adaptation)
        self.model.to(self.device).eval()
        self._refresh_runner()

    @classmethod
    def from_artifact(
        cls,
        artifact_path: str | Path,
        weights_dir: str | Path | None = None,
        device: str | None = None,
        allow_download: bool = False,
    ) -> ASTAudioClassificationPipeline:
        """Instantiate a base pipeline and load an adapted artifact."""
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_path)
        return pipe

    def predict(
        self,
        audio: np.ndarray,
        sample_rate: int,
        top_k: int = DEFAULT_TOP_K,
    ) -> dict[str, Any]:
        """Classify one clip. `audio` is a 1-D float array; `sample_rate` is the rate it was captured at."""
        duration = _check_inputs(audio, sample_rate, top_k)
        if top_k > len(self.labels):
            raise ValueError(f"top_k={top_k} exceeds the active label count ({len(self.labels)})")
        waveform = audio.astype(np.float32, copy=False)
        resampled = sample_rate != SAMPLE_RATE
        if resampled:
            waveform = _resample(waveform, sample_rate)
        logits = np.asarray(self._runner(waveform), dtype=np.float32)
        if logits.shape != (len(self.labels),):
            raise RuntimeError(f"backend returned logits {logits.shape}, expected ({len(self.labels)},)")
        if self.activation == "sigmoid":
            scores = 1.0 / (1.0 + np.exp(-logits))
        elif self.activation == "softmax":
            shifted = logits - np.max(logits)
            exp_scores = np.exp(shifted)
            scores = exp_scores / np.sum(exp_scores)
        else:
            raise RuntimeError(f"unsupported activation: {self.activation}")
        order = np.argsort(-scores)[:top_k]
        return {
            "predictions": [
                {"label": self.labels[int(i)], "index": int(i), "score": float(scores[i])} for i in order
            ],
            "activation": self.activation,
            "truncated": duration > MAX_AUDIO_SECONDS,
            "duration_seconds": duration,
            "window_seconds": MAX_AUDIO_SECONDS,
            "resampled": resampled,
            "input_sample_rate": sample_rate,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

**Module 3/3:** `src/ast_audio_classification_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Deterministic in-code sample audio data: pretrained demonstration tone and acoustic ecology dataset.

Nothing here is downloaded and nothing requires external dependencies beyond numpy,
ensuring zero network dataset dependencies and deterministic execution across clean-room environments.

Two separate label contexts live here and must not be conflated:
1. ``tutorial_tone``: 440 Hz sinusoidal waveform demonstrating pretrained 527-class AudioSet inference.
2. ``synthetic_audio_dataset``: 24 audio clips labelled with ``ADAPT_CLASSES``, the canonical
   Krause/Pijanowski tripartite acoustic ecology vocabulary:
   - ``geophony``: non-biological natural sounds (rain, wind, flowing water).
   - ``biophony``: non-human biological vocalizations (birdsong, insect chirps).
   - ``anthrophony``: human-generated mechanical sounds (engines, drones, machinery).
"""

from __future__ import annotations

import random
from collections.abc import Sequence
from typing import Any

import numpy as np

SAMPLE_RATE = 16_000
SAMPLE_DURATION = 3.0  # seconds; comfortably inside AST's 10.24 s window
MIN_DATASET_AUDIO_SECONDS = 0.025
MAX_DATASET_AUDIO_SECONDS = 10.24
TONE_HZ = 440.0
TONE_AMPLITUDE = 0.5

# Canonical Krause/Pijanowski acoustic ecology classification vocabulary
ADAPT_CLASSES: tuple[str, ...] = ("geophony", "biophony", "anthrophony")
DATASET_REPRESENTATION = "io.github.kurtvalcorza.dataset.audio.waveform-classification.v1"


def tutorial_tone(
    duration: float = SAMPLE_DURATION,
    sample_rate: int = SAMPLE_RATE,
    frequency: float = TONE_HZ,
    amplitude: float = TONE_AMPLITUDE,
) -> np.ndarray:
    """Deterministic single-tone audio waveform demonstrating pretrained AudioSet inference."""
    if not isinstance(sample_rate, int) or isinstance(sample_rate, bool) or sample_rate <= 0:
        raise ValueError("sample_rate must be a positive int")
    if not isinstance(duration, int | float) or isinstance(duration, bool) or duration <= 0:
        raise ValueError("duration must be positive")
    if not isinstance(frequency, int | float) or isinstance(frequency, bool) or frequency <= 0:
        raise ValueError("frequency must be positive")
    if not isinstance(amplitude, int | float) or isinstance(amplitude, bool) or not 0 <= amplitude <= 1:
        raise ValueError("amplitude must be in [0, 1]")
    t = np.linspace(0.0, duration, int(sample_rate * duration), endpoint=False, dtype=np.float32)
    return (amplitude * np.sin(2.0 * np.pi * frequency * t)).astype(np.float32)


def generate_audio_clip(
    index: int,
    class_idx: int,
    duration: float = SAMPLE_DURATION,
    sample_rate: int = SAMPLE_RATE,
    seed: int = 42,
) -> np.ndarray:
    """Generate a deterministic synthetic 1-D audio waveform characteristic of its acoustic class."""
    if isinstance(index, bool) or not isinstance(index, int) or index < 0:
        raise ValueError("index must be a non-negative int")
    valid_class_idx = (
        isinstance(class_idx, int)
        and not isinstance(class_idx, bool)
        and 0 <= class_idx < len(ADAPT_CLASSES)
    )
    if not valid_class_idx:
        raise ValueError(f"class_idx must be an int in [0, {len(ADAPT_CLASSES) - 1}]")
    if sample_rate != SAMPLE_RATE:
        raise ValueError(f"sample_rate must be {SAMPLE_RATE}")
    if not isinstance(duration, int | float) or isinstance(duration, bool):
        raise ValueError("duration must be numeric")
    if not MIN_DATASET_AUDIO_SECONDS <= duration <= MAX_DATASET_AUDIO_SECONDS:
        raise ValueError(
            f"duration must be in [{MIN_DATASET_AUDIO_SECONDS}, {MAX_DATASET_AUDIO_SECONDS}]"
        )
    if isinstance(seed, bool) or not isinstance(seed, int):
        raise ValueError("seed must be an int")
    n_samples = int(sample_rate * duration)
    t = np.linspace(0.0, duration, n_samples, endpoint=False, dtype=np.float32)
    rng = np.random.default_rng(seed=seed + index * 101 + class_idx * 17)

    if class_idx == 0:
        # Geophony: broadband turbulent noise with slow gentle envelope (simulating rain/wind)
        raw_noise = rng.standard_normal(n_samples).astype(np.float32)
        # 1-pole low-pass smooth filter
        alpha = 0.15
        filtered = np.zeros_like(raw_noise)
        acc = 0.0
        for i in range(n_samples):
            acc = alpha * raw_noise[i] + (1.0 - alpha) * acc
            filtered[i] = acc
        # Gentle swell envelope
        envelope = 0.7 + 0.3 * np.sin(2.0 * np.pi * 0.5 * t)
        signal = filtered * envelope
    elif class_idx == 1:
        # Biophony: rapid frequency-modulated chirp whistles (simulating bird song / biophonic calls)
        chirp_rate = 3.0 + (index % 3) * 1.5  # chirps per second
        f_center = 2800.0 + (index % 4) * 400.0
        f_dev = 1200.0
        # Phase modulation
        instantaneous_freq = f_center + f_dev * np.sin(2.0 * np.pi * chirp_rate * t)
        phase = 2.0 * np.pi * np.cumsum(instantaneous_freq) / sample_rate
        fundamental = np.sin(phase)
        harmonic = 0.35 * np.sin(2.0 * phase)
        # Pulsed amplitude gate
        gate = np.clip(np.sin(2.0 * np.pi * chirp_rate * t), 0.0, 1.0) ** 2
        signal = (fundamental + harmonic) * gate
    elif class_idx == 2:
        # Anthrophony: low-frequency harmonic drone with motor buzz (simulating engine/machinery)
        f0 = 120.0 + (index % 3) * 20.0  # 120 Hz fundamental
        harmonics = [
            (1.0, 0.45),
            (2.0, 0.30),
            (3.0, 0.20),
            (4.0, 0.15),
            (5.0, 0.10),
        ]
        drone = np.zeros(n_samples, dtype=np.float32)
        for mult, weight in harmonics:
            drone += weight * np.sin(2.0 * np.pi * (f0 * mult) * t)
        # Add motor stroke amplitude pulsation (15 Hz)
        engine_pulse = 0.75 + 0.25 * np.sin(2.0 * np.pi * 15.0 * t)
        signal = drone * engine_pulse
    # Normalize peak amplitude safely to [-0.85, 0.85]
    peak = np.max(np.abs(signal))
    if peak > 1e-6:
        signal = (signal / peak) * 0.85
    return signal.astype(np.float32)


def synthetic_audio_dataset(
    n_samples: int = 24,
    seed: int = 42,
    duration: float = SAMPLE_DURATION,
    sample_rate: int = SAMPLE_RATE,
) -> list[dict[str, Any]]:
    """Generate a balanced deterministic dataset across ADAPT_CLASSES."""
    if isinstance(n_samples, bool) or not isinstance(n_samples, int) or n_samples < len(ADAPT_CLASSES) * 2:
        raise ValueError(f"n_samples must be an int >= {len(ADAPT_CLASSES) * 2}")
    if n_samples % len(ADAPT_CLASSES):
        raise ValueError(f"n_samples must be divisible by {len(ADAPT_CLASSES)} for class balance")
    if isinstance(seed, bool) or not isinstance(seed, int):
        raise ValueError("seed must be an int")
    num_classes = len(ADAPT_CLASSES)
    records: list[dict[str, Any]] = []
    for idx in range(n_samples):
        class_idx = idx % num_classes
        waveform = generate_audio_clip(
            index=idx,
            class_idx=class_idx,
            duration=duration,
            sample_rate=sample_rate,
            seed=seed,
        )
        records.append(
            {
                "id": f"clip-{idx:03d}",
                "waveform": waveform,
                "sample_rate": sample_rate,
                "label": class_idx,
                "class_name": ADAPT_CLASSES[class_idx],
                "duration": round(float(duration), 4),
            }
        )
    return records


def split_dataset(
    records: Sequence[dict[str, Any]],
    val_fraction: float = 0.25,
    seed: int = 42,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Stratified train/validation split preserving balanced class distribution."""
    if not 0.0 < val_fraction < 1.0:
        raise ValueError("val_fraction must be strictly between 0 and 1")
    by_class: dict[int, list[dict[str, Any]]] = {}
    for r in records:
        by_class.setdefault(r["label"], []).append(r)

    rng = random.Random(seed)
    train_records: list[dict[str, Any]] = []
    val_records: list[dict[str, Any]] = []

    for label in sorted(by_class.keys()):
        class_list = list(by_class[label])
        if len(class_list) < 2:
            raise ValueError(f"class {label} requires at least 2 records for a disjoint split")
        rng.shuffle(class_list)
        n_val = min(len(class_list) - 1, max(1, round(len(class_list) * val_fraction)))
        val_records.extend(class_list[:n_val])
        train_records.extend(class_list[n_val:])

    rng.shuffle(train_records)
    rng.shuffle(val_records)
    return train_records, val_records


def validate_dataset(
    records: Sequence[dict[str, Any]],
    class_names: Sequence[str] = ADAPT_CLASSES,
) -> dict[str, Any]:
    """Validate that audio records strictly conform to the classification contract."""
    if not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise TypeError(f"records must be a sequence, got {type(records).__name__}")
    if len(records) < 2:
        raise ValueError(f"dataset requires at least 2 records, got {len(records)}")

    num_classes = len(class_names)
    if num_classes < 2:
        raise ValueError("class_names must contain at least 2 classes")
    if any(not isinstance(name, str) or not name.strip() for name in class_names):
        raise ValueError("class_names must contain non-empty strings")
    if len(set(class_names)) != num_classes:
        raise ValueError("class_names must be unique")
    class_counts = [0] * num_classes
    seen_ids: set[str] = set()

    for i, r in enumerate(records):
        if not isinstance(r, dict):
            raise TypeError(f"record {i} must be a dict, got {type(r).__name__}")
        if "waveform" not in r:
            raise KeyError(f"record {i} missing required key 'waveform'")
        if "label" not in r:
            raise KeyError(f"record {i} missing required key 'label'")
        if "id" not in r:
            raise KeyError(f"record {i} missing required key 'id'")
        if "sample_rate" not in r:
            raise KeyError(f"record {i} missing required key 'sample_rate'")
        if "class_name" not in r:
            raise KeyError(f"record {i} missing required key 'class_name'")

        record_id = r["id"]
        if not isinstance(record_id, str) or not record_id.strip():
            raise ValueError(f"record {i} id must be a non-empty string")
        if record_id in seen_ids:
            raise ValueError(f"record {i} duplicates id {record_id!r}")
        seen_ids.add(record_id)

        sample_rate = r["sample_rate"]
        if sample_rate != SAMPLE_RATE:
            raise ValueError(f"record {i} sample_rate must be {SAMPLE_RATE}, got {sample_rate!r}")

        waveform = r["waveform"]
        if not isinstance(waveform, np.ndarray):
            raise TypeError(f"record {i} waveform must be a np.ndarray, got {type(waveform).__name__}")
        if waveform.ndim != 1:
            raise ValueError(f"record {i} waveform must be 1-D mono, got shape {waveform.shape}")
        if waveform.dtype != np.float32:
            raise TypeError(f"record {i} waveform must be float32, got {waveform.dtype}")
        if not np.all(np.isfinite(waveform)):
            raise ValueError(f"record {i} waveform contains NaN or inf values")
        if waveform.size == 0:
            raise ValueError(f"record {i} waveform must not be empty")
        if np.max(np.abs(waveform)) > 1.0:
            raise ValueError(f"record {i} waveform samples must be in [-1, 1]")
        duration = waveform.size / sample_rate
        if not MIN_DATASET_AUDIO_SECONDS <= duration <= MAX_DATASET_AUDIO_SECONDS:
            raise ValueError(
                f"record {i} duration {duration:.4f} s outside "
                f"[{MIN_DATASET_AUDIO_SECONDS}, {MAX_DATASET_AUDIO_SECONDS}]"
            )

        label = r["label"]
        if isinstance(label, bool) or not isinstance(label, int):
            raise TypeError(f"record {i} label must be an int, got {type(label).__name__}")
        if not 0 <= label < num_classes:
            raise ValueError(f"record {i} label {label} outside valid range [0, {num_classes - 1}]")
        if r["class_name"] != class_names[label]:
            raise ValueError(
                f"record {i} class_name {r['class_name']!r} does not match "
                f"label {label} ({class_names[label]!r})"
            )

        class_counts[label] += 1

    missing_classes = [class_names[i] for i, count in enumerate(class_counts) if count == 0]
    if missing_classes:
        raise ValueError(f"dataset has no records for classes: {missing_classes}")

    return {
        "representation": DATASET_REPRESENTATION,
        "n_records": len(records),
        "num_classes": num_classes,
        "class_counts": {class_names[c]: class_counts[c] for c in range(num_classes)},
        "verdict": "accepted",
    }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `f826b80d2822…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `ASTAudioClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "ast-audioset",
  "modelId": "MIT/ast-finetuned-audioset-10-10-0.4593",
  "revision": "f826b80d28226b62986cc218e5cec390b1096902",
  "files": [
    {
      "path": "README.md",
      "bytes": 1165,
      "sha256": "dbc8ce1fc5abd1635d073640d8cb032901bfe264a5e06507f363f10a6112cfc8"
    },
    {
      "path": "config.json",
      "bytes": 26763,
      "sha256": "a93d525511d77e8ecc933d09674b85099815bbbb417c228a4edd655e252fb9ff"
    },
    {
      "path": "model.safetensors",
      "bytes": 346404948,
      "sha256": "ae0c1e2ad4e1381d851fa9bf298ba13ebc9c5a914cdee2dbe427a6583869924d"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 297,
      "sha256": "8d04ba5a9c6fca5d39d0de2b1fd05ecf79deb589fbba279728bbebac39934231"
    }
  ],
  "totalBytes": 346433173
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = ASTAudioClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Confirm the qualified runtime environment

The carried package verifies runtime dependency versions (`torch`, `torchaudio`, `transformers`) and prints the execution device alongside input ceilings. Look for confirmed versions and device.

In [ ]:
import platform
import numpy as np
import torch
import torchaudio
import transformers

print({
    'python': platform.python_version(),
    'torch': torch.__version__,
    'torchaudio': torchaudio.__version__,
    'transformers': transformers.__version__,
    'device': pipe.device,
    'ceilings': {
        'SAMPLE_RATE': SAMPLE_RATE,
        'MIN_AUDIO_SECONDS': MIN_AUDIO_SECONDS,
        'MAX_AUDIO_SECONDS': MAX_AUDIO_SECONDS,
        'MAX_INPUT_SECONDS': MAX_INPUT_SECONDS,
        'NUM_LABELS': NUM_LABELS,
    },
})

## 5. Demonstrate pretrained AudioSet capability

Before adapting to custom classes, this cell exercises the base model on a deterministic 440 Hz sinusoidal tone (or optional uploaded BYOD WAV). `predict` runs the clip through the pretrained 527-class head with independent sigmoid scores. Each score is **not a calibrated probability**, the scores do not sum to one, and no decision threshold is shipped. Look for top predicted AudioSet classes.

In [ ]:
import hashlib
import io
import wave
import numpy as np

USE_BYOD = False  # @param {type:"boolean"}
TONE_SECONDS = 3.0
TONE_HZ = 440.0
TONE_AMPLITUDE = 0.5

def decode_pcm_wav(payload):
    with wave.open(io.BytesIO(payload), 'rb') as handle:
        channels = handle.getnchannels()
        sample_width = handle.getsampwidth()
        decoded_rate = handle.getframerate()
        frames = handle.getnframes()
        compression = handle.getcomptype()
        raw = handle.readframes(frames)
    if compression != 'NONE':
        raise ValueError(f'compressed WAV is unsupported: {compression}')
    if sample_width == 1:
        samples = (np.frombuffer(raw, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0
    elif sample_width == 2:
        samples = np.frombuffer(raw, dtype='<i2').astype(np.float32) / 32768.0
    elif sample_width == 4:
        samples = np.frombuffer(raw, dtype='<i4').astype(np.float32) / 2147483648.0
    else:
        raise ValueError(f'{8 * sample_width}-bit PCM WAV is unsupported; convert to 16-bit PCM')
    if channels < 1 or samples.size % channels:
        raise ValueError('invalid WAV channel layout')
    mono = samples.reshape(-1, channels).mean(axis=1) if channels > 1 else samples
    return mono.astype(np.float32), decoded_rate

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('upload exactly one PCM WAV file')
    clip_name = next(iter(uploaded))
    audio, sample_rate = decode_pcm_wav(uploaded[clip_name])
    sample_kind = 'BYOD'
else:
    sample_rate = SAMPLE_RATE
    audio = tutorial_tone(duration=TONE_SECONDS, sample_rate=sample_rate, frequency=TONE_HZ, amplitude=TONE_AMPLITUDE)
    clip_name = f'synthetic_sine_{int(TONE_HZ)}hz_{int(TONE_SECONDS)}s.wav'
    sample_kind = 'synthetic'

audio_sha256 = hashlib.sha256(audio.tobytes()).hexdigest()
result = pipe.predict(audio, sample_rate=sample_rate, top_k=5)
print({
    'sample_kind': sample_kind,
    'name': clip_name,
    'samples': int(audio.shape[0]),
    'sample_rate': sample_rate,
    'sha256': audio_sha256[:16] + '...',
    'top_label': result['predictions'][0]['label'],
    'top_score': round(result['predictions'][0]['score'], 4),
    'activation': result['activation'],
    'truncated': result['truncated'],
})
for rank, item in enumerate(result['predictions'], start=1):
    print(f"  {rank:>2}. index {item['index']:>3}  score {item['score']:.4f}  {item['label']}")

## 6. Generate and validate the custom acoustic ecology dataset

The adaptation task targets the canonical Krause/Pijanowski tripartite acoustic ecology classification: `ADAPT_CLASSES = ('geophony', 'biophony', 'anthrophony')`. The dataset generator produces 24 16 kHz mono 3.0 s clips conforming to the owner-namespaced `io.github.kurtvalcorza.dataset.audio.waveform-classification.v1` representation; no stable `core.*` audio representation exists in the current DIMER fleet specification. `validate_dataset` checks audio integrity, duration limits, and class labels. The input manifest is written to `outputs/ast_audio_classification_input_manifest.json` along with a rejected over-long ceiling probe finding.

In [ ]:
import json
import os
import zipfile
from pathlib import PurePosixPath

os.makedirs('outputs', exist_ok=True)
USE_BYOD_DATASET = False  # @param {type:"boolean"}

if USE_BYOD_DATASET:
    from google.colab import files
    uploaded_dataset = files.upload()
    if len(uploaded_dataset) != 1:
        raise ValueError('upload exactly one ZIP dataset')
    dataset_name, dataset_bytes = next(iter(uploaded_dataset.items()))
    if not dataset_name.lower().endswith('.zip') or len(dataset_bytes) > 64 * 1024**2:
        raise ValueError('dataset must be one ZIP no larger than 64 MiB compressed')
    dataset_records = []
    with zipfile.ZipFile(io.BytesIO(dataset_bytes)) as archive:
        archive_members = archive.infolist()
        members = [member for member in archive_members if not member.is_dir()]
        if not members or len(members) > 100:
            raise ValueError('dataset ZIP must contain 1–100 files')
        if sum(member.file_size for member in members) > 256 * 1024**2:
            raise ValueError('dataset ZIP expands beyond 256 MiB')
        top_levels = set()
        for member in archive_members:
            path = PurePosixPath(member.filename)
            expected_parts = 1 if member.is_dir() else 2
            if path.is_absolute() or '..' in path.parts or len(path.parts) != expected_parts:
                raise ValueError(f'unsafe or invalid member path: {member.filename!r}')
            class_name = path.parts[0]
            if class_name not in ADAPT_CLASSES:
                raise ValueError(f'unknown class directory {class_name!r}; expected {ADAPT_CLASSES}')
            top_levels.add(class_name)
            if member.is_dir():
                continue
            if member.flag_bits & 0x1 or path.suffix.lower() != '.wav':
                raise ValueError(f'member must be an unencrypted PCM WAV: {member.filename!r}')
            waveform, record_rate = decode_pcm_wav(archive.read(member))
            dataset_records.append({
                'id': member.filename,
                'waveform': waveform,
                'sample_rate': record_rate,
                'label': ADAPT_CLASSES.index(class_name),
                'class_name': class_name,
            })
        if top_levels != set(ADAPT_CLASSES):
            raise ValueError(f'dataset ZIP must contain exactly these class directories: {ADAPT_CLASSES}')
    dataset_kind = 'BYOD'
else:
    dataset_records = synthetic_audio_dataset(n_samples=24, seed=42)
    dataset_kind = 'synthetic'

dataset_summary = validate_dataset(dataset_records, ADAPT_CLASSES)
if any(count < 2 for count in dataset_summary['class_counts'].values()):
    raise ValueError('dataset requires at least two records per class for a disjoint split')

input_manifest = validate_inputs(audio, sample_rate, top_k=5, names=[clip_name])
input_manifest['dataset_representation'] = dataset_summary['representation']
input_manifest['dataset_records'] = dataset_summary['n_records']
input_manifest['class_counts'] = dataset_summary['class_counts']

# Demonstrate validation rejection on an over-long input
try:
    validate_inputs(np.zeros(int((MAX_INPUT_SECONDS + 1) * SAMPLE_RATE), dtype=np.float32), SAMPLE_RATE)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'over-long-probe', 'verdict': 'rejected', 'message': str(exc)})

with open('outputs/ast_audio_classification_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)

print(json.dumps(input_manifest, indent=2))

## 7. Deterministic train/validation partition

The 24 records are partitioned into a 75% train set (18 clips, 6 per class) and a 25% held-out validation set (6 clips, 2 per class) using `split_dataset`. Stratification preserves exact class balance across splits.

In [ ]:
train_records, val_records = split_dataset(dataset_records, val_fraction=0.25, seed=42)
train_summary = validate_dataset(train_records, ADAPT_CLASSES)
val_summary = validate_dataset(val_records, ADAPT_CLASSES)

print({
    'split': 'stratified_75_25',
    'train_samples': len(train_records),
    'val_samples': len(val_records),
    'train_counts': train_summary['class_counts'],
    'val_counts': val_summary['class_counts'],
})

## 8. Dynamic re-heading and pre-adaptation baseline

`pipe.rehead(ADAPT_CLASSES)` dynamically swaps the 527-class AudioSet classifier head for a new 3-class linear head (`Linear(768, 3)`), and `pipe.freeze_backbone()` freezes the 85.5M backbone parameters so that only the classifier head (~3.8k parameters) is updated. Evaluating on the validation split before fine-tuning establishes the untrained baseline.

In [ ]:
pipe.rehead(ADAPT_CLASSES, seed=42)
frozen_params = pipe.freeze_backbone()

pre_eval = pipe.evaluate(val_records)
print({
    'stage': 'reheaded_pre_adaptation',
    'classes': pipe.labels,
    'frozen_backbone_parameters': frozen_params,
    'pre_adaptation_accuracy': pre_eval['accuracy'],
    'majority_baseline_accuracy': pre_eval['baseline']['majority_class_accuracy'],
})

## 9. In-process bounded fine-tuning loop

`pipe.finetune(...)` caches the frozen backbone features once, then executes bounded supervised adaptation using `torch.optim.AdamW` over 5 epochs (batch size 4, 25 optimizer steps total). The measured loss and held-out metrics are printed; the tutorial does not promise monotonic loss or a quality threshold on this synthetic sample.

In [ ]:
history = pipe.finetune(
    train_records=train_records,
    val_records=val_records,
    epochs=5,
    batch_size=4,
    learning_rate=1e-3,
    seed=42,
)

for epoch_data in history:
    print(f"Epoch {epoch_data['epoch']}/5: train_loss={epoch_data['train_loss']:.4f}  val_acc={epoch_data.get('val_accuracy', 0.0):.4f}")

## 10. Post-adaptation evaluation on held-out split

The adapted model is evaluated on the 6 held-out validation clips. `pipe.evaluate(val_records)` scores multiclass accuracy, macro-F1, per-class metrics, and compares against the majority-class baseline. The machine-readable report is written to `outputs/ast_audio_classification_evaluation_report.json` with verdict `sample-sanity`.

In [ ]:
eval_report = pipe.evaluate(val_records)

report_payload = {
    'task': 'multiclass acoustic ecology classification',
    'score_semantics': 'softmax probability distribution over target classes',
    'sample_kind': f'{dataset_kind}_heldout',
    'n_clips': len(val_records),
    'classes': list(ADAPT_CLASSES),
    'metrics': eval_report,
    'baseline': eval_report['baseline'],
    'verdict': 'sample-sanity',
    'reason': f"{len(val_records)} held-out clip(s) evaluated against majority baseline; not a benchmark claim",
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
}

with open('outputs/ast_audio_classification_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report_payload, handle, indent=2, ensure_ascii=False)

print(json.dumps(report_payload, indent=2))

## 11. Inference on unseen acoustic clip

Demonstrates the adapted pipeline on a newly synthesized unseen test clip (biophony chirp whistle). The predictions are exported to `outputs/ast_audio_classification_top_k.csv` with rank-ordered softmax probabilities.

In [ ]:
import csv

test_clip = generate_audio_clip(index=99, class_idx=1, duration=3.0, sample_rate=SAMPLE_RATE)
test_result = pipe.predict(test_clip, sample_rate=SAMPLE_RATE, top_k=3)

print({
    'test_clip': 'unseen_synthetic_biophony_99',
    'predicted_label': test_result['predictions'][0]['label'],
    'predicted_score': round(test_result['predictions'][0]['score'], 4),
    'activation': test_result['activation'],
})

with open('outputs/ast_audio_classification_top_k.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['clip', 'rank', 'index', 'label', 'score'])
    for rank, item in enumerate(test_result['predictions'], start=1):
        writer.writerow(['unseen_test_clip_99', rank, item['index'], item['label'], f"{item['score']:.6f}"])
        print(f"  {rank:>2}. index {item['index']:>2}  score {item['score']:.4f}  {item['label']}")

## 12. Export portable adapter artifact

`pipe.save_artifact('outputs/ast-audio-adapter-v1.pt')` persists the adapted weights, class vocabulary, adaptation configuration, and exact base-model identity to a reloadable classifier-head adapter. The frozen 85M-parameter backbone is deliberately not duplicated in this file and must be reconstructed from the pinned base revision.

In [ ]:
saved_artifact_path = pipe.save_artifact('outputs/ast-audio-adapter-v1.pt')
artifact_size = saved_artifact_path.stat().st_size
print({
    'saved_artifact': str(saved_artifact_path),
    'bytes': artifact_size,
    'format': ARTIFACT_FORMAT,
})

## 13. Clean reload and numeric verification

Verifies artifact portability across a clean boundary: instantiates a fresh pipeline and loads the adapter with safe `weights_only=True` deserialization. An assertion enforces exact numerical score equivalence.

In [ ]:
reloaded_pipe = ASTAudioClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
reloaded_pipe.load_artifact('outputs/ast-audio-adapter-v1.pt')

reloaded_result = reloaded_pipe.predict(test_clip, sample_rate=SAMPLE_RATE, top_k=3)
np.testing.assert_allclose(
    [p['score'] for p in reloaded_result['predictions']],
    [p['score'] for p in test_result['predictions']],
    rtol=1e-5,
    atol=1e-6,
)
print({
    'reload_verification': 'PASSED',
    'original_top_score': round(test_result['predictions'][0]['score'], 6),
    'reloaded_top_score': round(reloaded_result['predictions'][0]['score'], 6),
    'labels': reloaded_pipe.labels,
})

## 14. Export outputs and provenance bundle

Persists the complete machine-readable bundle (`manifest`, `report`, `result`, `top_k`, `adapter`) recording runtime metadata, base model revision, and license for traceability.

In [ ]:
payload = {
    'evaluation_report': report_payload,
    'input_manifest': input_manifest,
    'test_prediction': test_result,
    'training_history': history,
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'torchaudio': torchaudio.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/ast_audio_classification_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)

print(sorted(os.listdir('outputs')))

## Interpretation and limits

The adapted classifier predicts single-label softmax probability distributions over `ADAPT_CLASSES` (`geophony`, `biophony`, `anthrophony`). The baseline comparison measures performance relative to a trivial majority-class predictor on the held-out sample. Training is bounded to the classifier head with frozen backbone parameters, providing fast and reliable in-kernel adaptation without requiring GPU compute or external dependencies.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model, validate the demonstrated dataset contract, execute supervised head adaptation, evaluate metrics against a majority baseline, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or performance on real field recordings.

## References

- Repository README: https://github.com/kurtvalcorza/ast-audio-classification-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/ast-audio-classification-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/ast-audio-classification-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/MIT/ast-finetuned-audioset-10-10-0.4593
- Upstream code: https://github.com/YuanGongND/ast
- AST: Audio Spectrogram Transformer (Gong, Chung, Glass, 2021): https://arxiv.org/abs/2104.01778
- Acoustic ecology (Krause, 2008; Pijanowski et al., 2011)